Name: **Aryan Oberoi**
Roll no.: **I041**
Program: **B.Tech (Artificial Intelligence)**
Course: **Computer Networks**
Division: **I**
Batch: **B1**
Date of performance and submission: **29-08-2026**

---

# Project: Email Client Using SMTP and POP3
### Sending and receiving email within localhost using basic SMTP/POP3 logic

## 1. Aim

To design and implement a complete electronic mail system that runs entirely on
the loopback interface of a single host, consisting of:

1. an **SMTP server** implementing the mail submission protocol of RFC 5321,
2. a **POP3 server** implementing the mail access protocol of RFC 1939,
3. a **mail client** that speaks both protocols and provides a command-line and
   a graphical interface.

All three components are written directly on TCP sockets using Python's
`socket` module. The standard library's `smtplib`, `poplib` and `email` modules
are deliberately **not** used — the objective is to implement the protocol
logic, not to call a library that already contains it.

**Course outcome addressed —** CO5: *Demonstrate the functionalities of
application layer protocols such as HTTP, DNS, FTP, and SMTP for network
communication.*

## 2. Theory

### 2.1 Where email sits in the protocol stack

Electronic mail is an **application layer** service. It relies on the transport
layer below it for everything to do with reliability:

| Layer | Protocol | What it contributes here |
|---|---|---|
| Application | SMTP, POP3 | The command/response grammar and the mail semantics |
| Transport | **TCP** | Reliable, ordered, connection-oriented byte stream |
| Network | IP | Addressing and routing (here, loopback 127.0.0.1) |

TCP is used rather than UDP because a mail message must arrive complete, in
order, and exactly once. Neither SMTP nor POP3 contains any retransmission,
sequencing or checksum logic of its own — TCP already provides it. Every
session in this project therefore begins with the standard three-way handshake
before a single byte of protocol text is exchanged.

### 2.2 The three agents

A mail system is conventionally described with three roles:

* **MUA** — Mail User Agent. What the user sees. Composes and reads mail.
  *In this project:* `client/cli.py` and `client/gui.py`.
* **MTA** — Mail Transfer Agent. Accepts mail and delivers or relays it.
  *In this project:* `smtp_server.py`.
* **MAA** — Mail Access Agent. Hands stored mail to the user agent on request.
  *In this project:* `pop3_server.py`.

The essential asymmetry is this: **SMTP pushes, POP3 pulls.** Mail is pushed
towards its destination mailbox by SMTP, and is later pulled out of that
mailbox by POP3 when the recipient chooses to collect it. POP3 exists at all
because the recipient's machine is not guaranteed to be switched on when the
mail arrives.

### 2.3 SMTP — Simple Mail Transfer Protocol (RFC 5321)

SMTP is a text-based, connection-oriented protocol on **TCP port 25**. Every
command is a line ending in CRLF, and the server answers each one with a
**three-digit reply code**:

| Class | Meaning |
|---|---|
| **2yz** | Command accepted and completed |
| **3yz** | Command accepted, server is waiting for more input (only `354`) |
| **4yz** | Transient failure — the client may retry later |
| **5yz** | Permanent failure — retrying will not help |

The digits are structured, not arbitrary: the first gives the outcome, the
second the category (5 = syntax, 1 = information, 2 = connections), the third a
specific detail. A client can therefore act correctly on a code it has never
seen before by looking only at the first digit.

#### Command set implemented

| Command | Purpose | Typical replies |
|---|---|---|
| `HELO` | Identify the client (basic SMTP) | `250` |
| `EHLO` | Identify and request the capability list (ESMTP) | `250-` … `250` |
| `AUTH` | Authenticate (`LOGIN`, `PLAIN`) | `334`, `235`, `535` |
| `MAIL FROM:` | Begin a transaction; declare the envelope sender | `250`, `503`, `530` |
| `RCPT TO:` | Add one envelope recipient (repeatable) | `250`, `550`, `503` |
| `DATA` | Begin the message body | `354`, then `250` |
| `RSET` | Abort the transaction in progress | `250` |
| `NOOP` | Do nothing; keeps the connection alive | `250` |
| `VRFY` | Ask whether a mailbox exists | `250`, `550` |
| `QUIT` | Close the session | `221` |

#### The envelope versus the headers — an important distinction

`MAIL FROM` and `RCPT TO` form the **envelope**. This is what the server
actually routes on, and it is the only part the server reads. The `From:`,
`To:` and `Subject:` lines *inside* `DATA` are the **headers** — to SMTP they
are simply opaque bytes between `DATA` and the terminating dot.

The two need not agree. A message can carry `MAIL FROM:<attacker@evil>` on the
envelope and `From: principal@nmims.edu` in the header, and the recipient's
mail client will display the header. **This gap is the mechanism behind email
spoofing**, and it is why SPF, DKIM and DMARC were later added to check that
the envelope sender is authorised to use that domain.

#### End of data

Because there is no length field, the body is terminated by a line containing
a single period. A body that genuinely contains such a line must therefore
escape it: the sender prefixes an extra period to any line beginning with one
(**dot-stuffing**), and the receiver strips it back off. Without this, a message
containing a lone `.` line would silently truncate on delivery.

### 2.4 POP3 — Post Office Protocol version 3 (RFC 1939)

POP3 runs on **TCP port 110** and is markedly simpler than SMTP. There are no
numeric codes at all — every response begins with either `+OK` or `-ERR`.

Its defining feature is that a session passes through **exactly three states**:

```
        connect
           │
           ▼
  ┌──────────────────┐   USER/PASS or APOP succeeds
  │  AUTHORIZATION   │────────────────────────────────┐
  └──────────────────┘                                │
       client identifies itself;                      ▼
       server locks the maildrop          ┌──────────────────────┐
                                          │     TRANSACTION      │
                                          │  STAT LIST UIDL RETR │
                                          │  TOP  DELE RSET NOOP │
                                          └──────────┬───────────┘
                                                     │ QUIT
                                                     ▼
                                          ┌──────────────────────┐
                                          │       UPDATE         │
                                          │ messages marked DELE │
                                          │  are actually removed│
                                          └──────────────────────┘
```

**`DELE` does not delete anything.** It sets a flag. The message is removed only
when the session reaches the UPDATE state, which happens on `QUIT`. `RSET`
clears every flag, and if the connection drops before `QUIT` the maildrop is
left completely untouched.

This is a deliberate design choice, not an oversight: a client that crashes
half way through collecting its mail loses nothing. Section 6 below
demonstrates both outcomes.

A second consequence of the state model: **message numbers are assigned once,
when the session opens, and never shift.** Deleting message 1 does not renumber
message 2 — message 1 simply becomes inaccessible for the remainder of the
session. `UIDL` provides identifiers that are stable *across* sessions, which
is what a client uses to implement "leave mail on server".

| Command | Purpose |
|---|---|
| `USER` / `PASS` | Authenticate with a cleartext password |
| `APOP` | Authenticate with `MD5(banner + secret)` — the password never crosses the wire |
| `STAT` | Message count and total size |
| `LIST` | Per-message sizes, without downloading anything |
| `UIDL` | Unique ids, stable across sessions |
| `RETR n` | Download message *n* in full |
| `TOP n m` | Headers plus the first *m* body lines — enough for a list view |
| `DELE n` | Mark message *n* for deletion |
| `RSET` | Unmark everything |
| `QUIT` | Enter UPDATE, commit deletions, close |

### 2.5 Comparison: SMTP, POP3 and IMAP

| | **SMTP** | **POP3** | **IMAP** |
|---|---|---|---|
| RFC | 5321 | 1939 | 3501 |
| Default port | 25 (587, 465) | 110 (995) | 143 (993) |
| Direction | **Push** | **Pull** | **Pull** |
| Role | Transfer / submission | Access | Access |
| Response format | 3-digit codes | `+OK` / `-ERR` | Tagged `OK`/`NO`/`BAD` |
| Where mail lives | In transit | Downloaded to the client | Stays on the server |
| Multiple devices | n/a | Poor — each download is separate | Designed for it |
| Server folders | n/a | Inbox only | Arbitrary folder hierarchy |
| Partial fetch | n/a | Only `TOP` | Any part of any message |
| Offline use | n/a | Excellent | Requires sync or caching |
| Complexity | Moderate | **Very low** | High |

POP3's simplicity is the reason it is chosen for this project: the entire
protocol fits in a few hundred lines and its three-state model is small enough
to be demonstrated exhaustively. IMAP is the modern default in practice because
mail is now read from several devices, and POP3's download-and-delete model
serves that badly.

### 2.6 Ports used in this project

| | This project | Standard |
|---|---|---|
| SMTP | **1025** | 25 (587 submission, 465 implicit TLS) |
| POP3 | **1110** | 110 (995 implicit TLS) |

Ports below 1024 are privileged and require root to bind, so unprivileged ports
are used instead. The protocol logic is identical — only the port number
differs. This also illustrates a useful point: *"port 25 is SMTP"* is a
convention recorded in the IANA registry, not a property of the protocol
itself.

## 3. System Architecture

```
        ┌──────────────────────────────────────────┐
        │       Mail User Agent (MUA)              │
        │     client/cli.py    client/gui.py       │
        └────────┬─────────────────────────┬───────┘
                 │ SMTPClient              │ POP3Client
                 │ (raw sockets)           │ (raw sockets)
                 ▼                         ▼
      ┌────────────────────┐    ┌────────────────────┐
      │  127.0.0.1 : 1025  │    │  127.0.0.1 : 1110  │
      │   smtp_server.py   │    │   pop3_server.py   │
      │      (MTA)         │    │      (MAA)         │
      │    RFC 5321        │    │    RFC 1939        │
      └─────────┬──────────┘    └─────────┬──────────┘
                │  writes                 │  reads
                └────────►  store/  ◄─────┘
                         <user>/*.eml
```

The SMTP server writes one file per message into the recipient's mailbox
directory; the POP3 server later reads that same file back out. Nothing else
connects the two servers. This mirrors a real deployment, where an MTA spools
messages to disk and an entirely separate access server hands them to the user.

### Module breakdown

| File | Responsibility |
|---|---|
| `common/wire.py` | The only module that touches a socket: CRLF line reading, dot-stuffing |
| `common/trace.py` | The `C:` / `S:` protocol tracer; masks credentials before logging |
| `common/mailstore.py` | Mailboxes on disk, maildrop locking, message numbering |
| `common/mime.py` | Builds and parses RFC 5322 / MIME messages by hand |
| `common/users.py` | Accounts, salted password hashing |
| `smtp_server.py` | SMTP state machine and delivery |
| `pop3_server.py` | POP3 three-state machine |
| `client/mail_client.py` | `SMTPClient` and `POP3Client` |

### Handling concurrency

Each server runs an `accept()` loop and hands every accepted connection to its
own thread. This matters for POP3 in particular, where a session holds an
exclusive lock on the maildrop for its whole duration — a second client
attempting to open the same mailbox is refused with `-ERR`, exactly as RFC 1939
requires, rather than being shown a mailbox that changes underneath it.

## 4. Implementation

The cells below start both servers inside this notebook and drive them with the
real client. Every `C:` line was sent by the client and every `S:` line was sent
by the server — this is the actual protocol conversation, not a printed
summary.

In [1]:
import sys, tempfile, warnings
from pathlib import Path

# The notebook lives beside the project modules.
PROJECT = Path.cwd()
sys.path.insert(0, str(PROJECT))

import config
from common.mailstore import MailStore
from common.users import UserDirectory
from common.trace import Tracer
from common import mime
from smtp_server import SMTPServer
from pop3_server import POP3Server
from client.mail_client import SMTPClient, POP3Client, SMTPError, POP3Error

# A throwaway mail store, so running the notebook does not disturb store/.
workdir = Path(tempfile.mkdtemp(prefix="cn-notebook-"))
store = MailStore(workdir / "store")
users = UserDirectory(workdir / "users.json")

smtp_server = SMTPServer("127.0.0.1", 0, store=store, users=users,
                         verbose=False, log_to_file=False)
pop_server  = POP3Server("127.0.0.1", 0, store=store, users=users,
                         verbose=False, log_to_file=False)
SMTP_PORT = smtp_server.start()
POP_PORT  = pop_server.start()

# One tracer, shared by every session below, printing to the notebook output.
tracer = Tracer("notebook", to_console=True, to_file=False, color=False)

print(f"SMTP server listening on 127.0.0.1:{SMTP_PORT}")
print(f"POP3 server listening on 127.0.0.1:{POP_PORT}")
print(f"Mailboxes: {', '.join(users.usernames())}")

SMTP server listening on 127.0.0.1:53041
POP3 server listening on 127.0.0.1:53042
Mailboxes: aryan, friend, prof


### 4.1 SMTP — connection, capability negotiation and authentication

The server speaks first with a `220` greeting; the client must read before it
may send. `EHLO` then asks what the server supports, and the reply is
multi-line — every line but the last has a hyphen after the code, which is how
the client knows more is coming.

In [2]:
smtp = SMTPClient("127.0.0.1", SMTP_PORT, tracer)
smtp.connect()          # reads the 220 greeting
smtp.ehlo("nmims.client.local")

print()
print("Capabilities advertised by the server:")
for capability in smtp.capabilities:
    print("   ", capability)

[03:51:25.671] [notebook] *: connecting to SMTP 127.0.0.1:53041


[03:51:25.672] [notebook] S: 220 localhost Simple SMTP Service Ready (CN Project)


[03:51:25.672] [notebook] C: EHLO nmims.client.local


[03:51:25.673] [notebook] S: 250-localhost Hello nmims.client.local [127.0.0.1]


[03:51:25.673] [notebook] S: 250-SIZE 10485760


[03:51:25.673] [notebook] S: 250-8BITMIME


[03:51:25.673] [notebook] S: 250-AUTH LOGIN PLAIN


[03:51:25.674] [notebook] S: 250 HELP



Capabilities advertised by the server:
    SIZE 10485760
    8BITMIME
    AUTH LOGIN PLAIN
    HELP


`AUTH LOGIN` is a two-step challenge. The server sends `334 VXNlcm5hbWU6` —
base64 for `Username:` — and the client replies with the base64 username; the
same follows for the password.

Note that the trace shows `<credential hidden>`. The tracer masks credentials
before writing them anywhere, so the log files submitted with this project do
not contain a working password. **The wire itself is not protected** — base64
is an encoding, not encryption, and section 7 shows the password being read
straight out of a packet capture.

In [3]:
smtp.login("aryan", "aryan123")

[03:51:25.678] [notebook] C: AUTH LOGIN


[03:51:25.678] [notebook] S: 334 VXNlcm5hbWU6


[03:51:25.678] [notebook] C: <credential hidden>


[03:51:25.679] [notebook] S: 334 UGFzc3dvcmQ6


[03:51:25.679] [notebook] C: <credential hidden>


[03:51:25.679] [notebook] S: 235 2.7.0 Authentication successful


[03:51:25.679] [notebook] *: authenticated as aryan


True

### 4.2 SMTP — sending a message

A complete transaction: `MAIL FROM` declares the envelope sender, one `RCPT TO`
per recipient, then `DATA`. The server answers `354`, the message follows, and
a lone `.` on its own line ends it.

In [4]:
queue_id = smtp.send_mail(
    "aryan@localhost",
    ["prof@localhost"],
    "Lab 6 submission",
    "Good evening sir,\n\n"
    "Please find my Computer Networks lab submission below.\n\n"
    "Regards,\nAryan Oberoi (I041)")

print()
print("Server accepted the message:", queue_id)

[03:51:25.684] [notebook] C: MAIL FROM:<aryan@localhost>


[03:51:25.685] [notebook] S: 250 2.1.0 Sender <aryan@localhost> OK


[03:51:25.685] [notebook] C: RCPT TO:<prof@localhost>


[03:51:25.685] [notebook] S: 250 2.1.5 Recipient <prof@localhost> OK


[03:51:25.685] [notebook] C: DATA


[03:51:25.686] [notebook] S: 354 End data with <CRLF>.<CRLF>


[03:51:25.686] [notebook] *: <sending message: 405 octets, 10 lines>


[03:51:25.686] [notebook] C: .


[03:51:25.687] [notebook] S: 250 2.0.0 OK: queued as 1787955685-8766-000001



Server accepted the message: 2.0.0 OK: queued as 1787955685-8766-000001


### 4.3 SMTP — one message, several recipients

`DATA` transfers the message **once**, but `RCPT TO` is issued once per
recipient and the server writes a separate copy into each mailbox. This is why
mailing to fifty people does not require fifty transfers of the body.

In [5]:
smtp.send_mail("aryan@localhost",
               ["prof@localhost", "friend@localhost"],
               "Group update",
               "Sharing this with both of you.")

print()
for user in users.usernames():
    count, size = store.message_count(user)
    print(f"  {user:<8} {count} message(s), {size} octets")

[03:51:25.690] [notebook] C: MAIL FROM:<aryan@localhost>


[03:51:25.690] [notebook] S: 250 2.1.0 Sender <aryan@localhost> OK


[03:51:25.690] [notebook] C: RCPT TO:<prof@localhost>


[03:51:25.691] [notebook] S: 250 2.1.5 Recipient <prof@localhost> OK


[03:51:25.691] [notebook] C: RCPT TO:<friend@localhost>


...

[03:51:25.692] [notebook] S: 354 End data with <CRLF>.<CRLF>


[03:51:25.692] [notebook] *: <sending message: 346 octets, 10 lines>


[03:51:25.692] [notebook] C: .


[03:51:25.693] [notebook] S: 250 2.0.0 OK: queued as 1787955685-8766-000003



  aryan    0 message(s), 0 octets
  friend   1 message(s), 498 octets
  prof     2 message(s), 1056 octets


### 4.4 MIME — attaching a file

SMTP transports text. To carry a binary file, MIME (RFC 2045/2046) layers
structure inside the message body: `Content-Type: multipart/mixed` with a
boundary string, each part separated by `--boundary`, and binary parts
base64-encoded so they survive a text-only transport.

`common/mime.py` builds and parses this by hand rather than using Python's
`email` package.

In [6]:
attachment = workdir / "rtt_readings.csv"
attachment.write_text("trial,rtt_ms\n1,0.41\n2,0.38\n3,0.44\n")

smtp.send_mail("aryan@localhost", ["prof@localhost"],
               "Loopback RTT readings",
               "Sir, the measurements are attached.",
               attachments=[attachment])
smtp.quit()

# Inspect the message that was actually written to the mailbox.
number, filename, size = store.list_messages("prof")[-1]
raw = store.read_message("prof", filename)

print()
print("--- the message as stored on disk (first 26 lines) ---")
for line in raw.decode().split("\r\n")[:26]:
    print(line)

[03:51:25.696] [notebook] C: MAIL FROM:<aryan@localhost>


[03:51:25.697] [notebook] S: 250 2.1.0 Sender <aryan@localhost> OK


[03:51:25.697] [notebook] C: RCPT TO:<prof@localhost>


[03:51:25.697] [notebook] S: 250 2.1.5 Recipient <prof@localhost> OK


[03:51:25.698] [notebook] C: DATA


...

[03:51:25.699] [notebook] C: .


[03:51:25.700] [notebook] S: 250 2.0.0 OK: queued as 1787955685-8766-000004


[03:51:25.700] [notebook] C: QUIT


[03:51:25.700] [notebook] S: 221 2.0.0 localhost closing connection



--- the message as stored on disk (first 26 lines) ---
Received: from nmims.client.local (127.0.0.1)
	by localhost (CN-Project SMTP) with ESMTP;
	for <prof@localhost>; Sat, 29 Aug 2026 03:51:25 +0530
From: aryan@localhost
To: prof@localhost
Subject: Loopback RTT readings
Date: Sat, 29 Aug 2026 03:51:25 +0530
Message-ID: <1787955685697.y5g0sun706e0@localhost>
MIME-Version: 1.0
X-Mailer: CN-Project-MailClient/1.0 (I041)
Content-Type: multipart/mixed; boundary="----=_CNProject_E4skI7rKqcw05jWQF0QJxZyp"

------=_CNProject_E4skI7rKqcw05jWQF0QJxZyp
Content-Type: text/plain; charset="utf-8"
Content-Transfer-Encoding: 8bit

Sir, the measurements are attached.
------=_CNProject_E4skI7rKqcw05jWQF0QJxZyp
Content-Type: text/csv; name="rtt_readings.csv"
Content-Transfer-Encoding: base64
Content-Disposition: attachment; filename="rtt_readings.csv"

dHJpYWwscnR0X21zCjEsMC40MQoyLDAuMzgKMywwLjQ0Cg==
------=_CNProject_E4skI7rKqcw05jWQF0QJxZyp--



### 4.5 SMTP — error handling

The reply code carries real information, and the distinction between `500` and
`503` is the state machine talking:

* **`500`** — the command does not exist.
* **`503`** — the command exists but is not legal *at this point in the
  conversation*.
* **`550`** — the command is legal and well-formed, but the mailbox does not
  exist. Permanent: retrying will not help.

In [7]:
probe = SMTPClient("127.0.0.1", SMTP_PORT, tracer)
probe.connect()
probe.helo("nmims.client.local")

for description, command in [
        ("a command that does not exist",            "TELEPORT now"),
        ("RCPT before MAIL - wrong order",           "RCPT TO:<prof@localhost>"),
        ("MAIL with malformed syntax",               "MAIL aryan@localhost"),
]:
    print(f"\n--- {description} ---")
    probe._send(command)
    code_, text = probe._read_response()
    print(f"    -> {code_} {' '.join(text)}")

print("\n--- a valid transaction addressed to a mailbox that does not exist ---")
probe._send("MAIL FROM:<aryan@localhost>"); probe._read_response()
probe._send("RCPT TO:<nosuchuser@localhost>")
code_, text = probe._read_response()
print(f"    -> {code_} {' '.join(text)}")
probe.quit()

[03:51:25.704] [notebook] *: connecting to SMTP 127.0.0.1:53041


[03:51:25.704] [notebook] S: 220 localhost Simple SMTP Service Ready (CN Project)


[03:51:25.704] [notebook] C: HELO nmims.client.local


[03:51:25.704] [notebook] S: 250 localhost Hello nmims.client.local [127.0.0.1]



--- a command that does not exist ---
[03:51:25.705] [notebook] C: TELEPORT now


...

[03:51:25.706] [notebook] S: 250 2.1.0 Sender <aryan@localhost> OK


[03:51:25.706] [notebook] C: RCPT TO:<nosuchuser@localhost>


[03:51:25.706] [notebook] S: 550 5.1.1 <nosuchuser@localhost>: no such user here


    -> 550 5.1.1 <nosuchuser@localhost>: no such user here
[03:51:25.706] [notebook] C: QUIT


[03:51:25.707] [notebook] S: 221 2.0.0 localhost closing connection


## 5. Receiving mail with POP3

The greeting carries a timestamp banner in angle brackets — that is the
challenge used by `APOP`, and it must be unique per session.

In [8]:
pop = POP3Client("127.0.0.1", POP_PORT, tracer)
pop.connect()
pop.login("prof", "prof123")

count, octets = pop.stat()
print()
print(f"STAT -> {count} message(s), {octets} octets total")

[03:51:25.710] [notebook] *: connecting to POP3 127.0.0.1:53042


[03:51:25.711] [notebook] S: +OK POP3 server ready (CN Project) <8766.1787955685711@localhost>


[03:51:25.711] [notebook] C: USER prof


[03:51:25.711] [notebook] S: +OK user prof accepted, send PASS


[03:51:25.711] [notebook] C: PASS <credential hidden>


[03:51:25.712] [notebook] S: +OK maildrop has 3 message(s) (1966 octets)


[03:51:25.712] [notebook] *: logged in as prof


[03:51:25.712] [notebook] C: STAT


[03:51:25.712] [notebook] S: +OK 3 1966



STAT -> 3 message(s), 1966 octets total


`LIST` gives per-message sizes without downloading anything, and `UIDL` gives
identifiers that stay the same across sessions — which is how a mail client
implements "leave mail on server" and still knows what it has already seen.

In [9]:
listing = pop.list()
uids = dict(pop.uidl())

print()
print(f"{'#':>3}  {'Size':>7}  Unique-ID")
for number, size in listing:
    print(f"{number:>3}  {size:>7}  {uids[number]}")

[03:51:25.715] [notebook] C: LIST


[03:51:25.715] [notebook] S: +OK 3 message(s) (1966 octets)


[03:51:25.715] [notebook] *: <multi-line response: 21 octets, 3 lines>


[03:51:25.715] [notebook] S: .


[03:51:25.716] [notebook] C: UIDL


[03:51:25.716] [notebook] S: +OK unique-id listing follows


[03:51:25.716] [notebook] *: <multi-line response: 78 octets, 3 lines>


[03:51:25.716] [notebook] S: .



  #     Size  Unique-ID
  1      560  1787955685-8766-000001
  2      496  1787955685-8766-000002
  3      910  1787955685-8766-000004


`TOP n m` returns the headers plus the first *m* lines of the body. This is how
a client renders a message list quickly: it needs the subject lines, not the
megabytes of attachments.

In [10]:
print()
for number, size in listing:
    headers = mime.parse_message(pop.top(number, 0))   # headers only
    print(f"  {number}.  {headers.sender:<20} {headers.subject}")


[03:51:25.720] [notebook] C: TOP 1 0


[03:51:25.720] [notebook] S: +OK top of message 1 follows


[03:51:25.720] [notebook] *: <multi-line response: 450 octets, 13 lines>


[03:51:25.720] [notebook] S: .


  1.  aryan@localhost      Lab 6 submission
[03:51:25.720] [notebook] C: TOP 2 0


...

  2.  aryan@localhost      Group update
[03:51:25.721] [notebook] C: TOP 3 0


[03:51:25.722] [notebook] S: +OK top of message 3 follows


[03:51:25.722] [notebook] *: <multi-line response: 463 octets, 12 lines>


[03:51:25.722] [notebook] S: .


  3.  aryan@localhost      Loopback RTT readings


`RETR n` downloads a message in full. Note the `Received:` header at the top —
the SMTP server stamped it during delivery. On a real message, reading these
headers from the bottom upwards reconstructs the exact path the message took
between servers.

In [11]:
message = mime.parse_message(pop.retr(1))

print()
print("From    :", message.sender)
print("To      :", message.recipients)
print("Subject :", message.subject)
print("Date    :", message.date)
print("Received:", message.header("received"))
print("-" * 70)
print(message.body)

[03:51:25.725] [notebook] C: RETR 1


[03:51:25.726] [notebook] S: +OK 560 octets


[03:51:25.726] [notebook] *: <multi-line response: 560 octets, 19 lines>


[03:51:25.726] [notebook] S: .



From    : aryan@localhost
To      : prof@localhost
Subject : Lab 6 submission
Date    : Sat, 29 Aug 2026 03:51:25 +0530
Received: from nmims.client.local (127.0.0.1) by localhost (CN-Project SMTP) with ESMTP; for <prof@localhost>; Sat, 29 Aug 2026 03:51:25 +0530
----------------------------------------------------------------------
Good evening sir,

Please find my Computer Networks lab submission below.

Regards,
Aryan Oberoi (I041)



Retrieving the message that carries the attachment: the client parses the
multipart structure and base64-decodes the part back into the original bytes.

In [12]:
with_file = mime.parse_message(pop.retr(3))

print()
print("Subject :", with_file.subject)
print("Body    :", with_file.body.strip())
for item in with_file.attachments:
    print(f"Attached: {item.filename}  ({item.size} bytes, {item.content_type})")
    print("-" * 70)
    print(item.data.decode())

[03:51:25.729] [notebook] C: RETR 3


[03:51:25.729] [notebook] S: +OK 910 octets


[03:51:25.730] [notebook] *: <multi-line response: 910 octets, 24 lines>


[03:51:25.730] [notebook] S: .



Subject : Loopback RTT readings
Body    : Sir, the measurements are attached.
Attached: rtt_readings.csv  (34 bytes, text/csv)
----------------------------------------------------------------------
trial,rtt_ms
1,0.41
2,0.38
3,0.44



## 6. The POP3 three-state model in action

This is the part of POP3 most worth demonstrating.

**`DELE` does not delete.** It marks. The message is removed only when the
session enters the UPDATE state, which happens on `QUIT`. First, what happens
when the connection is closed *without* `QUIT`:

In [13]:
print("Messages on disk before:", len(store.list_messages("prof")))

pop.dele(1)
print()
print("After DELE 1, STAT reports:", pop.stat()[0],
      "message(s) - the marked one is hidden from the client")

pop.close()          # deliberately close WITHOUT sending QUIT

print("Messages still on disk :", len(store.list_messages("prof")),
      "  <- nothing was deleted; UPDATE state was never reached")

Messages on disk before: 3
[03:51:25.732] [notebook] C: DELE 1


[03:51:25.733] [notebook] S: +OK message 1 marked deleted



[03:51:25.733] [notebook] C: STAT


[03:51:25.733] [notebook] S: +OK 2 1406


After DELE 1, STAT reports: 2 message(s) - the marked one is hidden from the client
Messages still on disk : 3   <- nothing was deleted; UPDATE state was never reached


Now the same sequence, but ending with `QUIT`. `RSET` is shown first, undoing a
mark, before the deletion is finally committed.

In [14]:
pop = POP3Client("127.0.0.1", POP_PORT, tracer)
pop.connect()
pop.login("prof", "prof123")

pop.dele(1)
print("\nAfter DELE 1 :", pop.stat()[0], "message(s) visible")
pop.rset()
print("After RSET   :", pop.stat()[0], "message(s) visible - the mark was undone")

pop.dele(1)
print("After DELE 1 again:", pop.stat()[0], "message(s) visible")

before = len(store.list_messages("prof"))
pop.quit()                       # QUIT -> UPDATE state -> actual deletion
after = len(store.list_messages("prof"))

print(f"\nOn disk before QUIT: {before}")
print(f"On disk after QUIT : {after}   <- the UPDATE state removed it")

[03:51:25.736] [notebook] *: connecting to POP3 127.0.0.1:53042


[03:51:25.737] [notebook] S: +OK POP3 server ready (CN Project) <8766.1787955685737@localhost>


[03:51:25.737] [notebook] C: USER prof


[03:51:25.737] [notebook] S: +OK user prof accepted, send PASS


[03:51:25.737] [notebook] C: PASS <credential hidden>


...

[03:51:25.740] [notebook] C: STAT


[03:51:25.741] [notebook] S: +OK 2 1406


After DELE 1 again: 2 message(s) visible
[03:51:25.741] [notebook] C: QUIT


[03:51:25.742] [notebook] S: +OK localhost POP3 server signing off (1 message(s) deleted)



On disk before QUIT: 3
On disk after QUIT : 2   <- the UPDATE state removed it


### 6.1 Message numbers do not shift after `DELE`

Deleting message 1 does not renumber message 2. Within a session the numbering
is fixed; the deleted slot simply becomes inaccessible.

In [15]:
pop = POP3Client("127.0.0.1", POP_PORT, tracer)
pop.connect()
pop.login("prof", "prof123")

pop.dele(1)
print()
try:
    pop.retr(1)
except POP3Error as exc:
    print("RETR 1 ->", exc, "  (marked deleted)")

kept = mime.parse_message(pop.retr(2))
print("RETR 2 -> still message 2:", kept.subject, "  (it did not slide into slot 1)")

pop.rset()      # undo, so nothing is actually lost
pop.quit()

[03:51:25.745] [notebook] *: connecting to POP3 127.0.0.1:53042


[03:51:25.745] [notebook] S: +OK POP3 server ready (CN Project) <8766.1787955685745@localhost>


[03:51:25.745] [notebook] C: USER prof


[03:51:25.746] [notebook] S: +OK user prof accepted, send PASS


[03:51:25.746] [notebook] C: PASS <credential hidden>


...

RETR 2 -> still message 2: Loopback RTT readings   (it did not slide into slot 1)
[03:51:25.748] [notebook] C: RSET


[03:51:25.748] [notebook] S: +OK maildrop has 2 message(s) (1406 octets)


[03:51:25.748] [notebook] C: QUIT


[03:51:25.749] [notebook] S: +OK localhost POP3 server signing off (0 message(s) deleted)


'localhost POP3 server signing off (0 message(s) deleted)'

### 6.2 The maildrop is locked for the duration of a session

RFC 1939 requires exclusive access, so a client is never shown a mailbox that
changes underneath it mid-session.

In [16]:
first = POP3Client("127.0.0.1", POP_PORT, tracer)
first.connect()
first.login("prof", "prof123")

second = POP3Client("127.0.0.1", POP_PORT, tracer)
second.connect()
print()
try:
    second.login("prof", "prof123")
except POP3Error as exc:
    print("Second session refused ->", exc)
second.close()

first.quit()
print("After the first session ends, the maildrop is available again.")

[03:51:25.752] [notebook] *: connecting to POP3 127.0.0.1:53042


[03:51:25.753] [notebook] S: +OK POP3 server ready (CN Project) <8766.1787955685753@localhost>


[03:51:25.753] [notebook] C: USER prof


[03:51:25.753] [notebook] S: +OK user prof accepted, send PASS


[03:51:25.754] [notebook] C: PASS <credential hidden>


...

[03:51:25.755] [notebook] C: PASS <credential hidden>


[03:51:25.755] [notebook] S: -ERR maildrop already locked by another session


Second session refused -> maildrop already locked by another session
[03:51:25.756] [notebook] C: QUIT


[03:51:25.756] [notebook] S: +OK localhost POP3 server signing off (0 message(s) deleted)


After the first session ends, the maildrop is available again.


### 6.3 Authenticating without sending the password: `APOP`

`APOP` sends `MD5(banner + shared secret)`. The secret itself never crosses the
wire, so a packet capture reveals nothing usable.

Its weakness is the other side of the same coin: to verify the digest, the
server must store the password in plaintext. That trade — wire security bought
with an insecure password database — is why APOP was abandoned in favour of
running the whole session inside TLS.

In [17]:
apop_client = POP3Client("127.0.0.1", POP_PORT, tracer)
apop_client.connect()
apop_client.apop("prof", "prof123")
print()
print("Logged in via APOP; STAT ->", apop_client.stat())
apop_client.quit()

[03:51:25.759] [notebook] *: connecting to POP3 127.0.0.1:53042


[03:51:25.759] [notebook] S: +OK POP3 server ready (CN Project) <8766.1787955685759@localhost>


[03:51:25.759] [notebook] C: APOP <credential hidden>


[03:51:25.760] [notebook] S: +OK maildrop has 2 message(s) (1406 octets)



[03:51:25.760] [notebook] C: STAT


[03:51:25.761] [notebook] S: +OK 2 1406


Logged in via APOP; STAT -> (2, 1406)
[03:51:25.761] [notebook] C: QUIT


[03:51:25.761] [notebook] S: +OK localhost POP3 server signing off (0 message(s) deleted)


'localhost POP3 server signing off (0 message(s) deleted)'

## 7. Analysis of the traffic on the wire

The trace above is what the *program* believes it sent. A packet capture shows
what actually crossed the interface. The full procedure is in
`docs/wireshark_capture.md`; the essentials:

```bash
sudo tcpdump -i lo0 -s 0 -w cn_smtp_pop.pcap 'tcp port 1025 or tcp port 1110'
python3 run_demo.py --keep
```

Then open the capture in Wireshark, filter on
`tcp.port == 1025 || tcp.port == 1110`, and use **Follow → TCP Stream** to see
the whole conversation as readable text.

**One practical detail:** Wireshark's SMTP and POP dissectors are registered
against ports 25 and 110. On 1025/1110 the packets appear as plain TCP, so use
right-click → **Decode As…** to map the port to SMTP or POP. This is itself
instructive — it shows that the port-to-protocol association is a registry
convention, not a property of the protocol.

### What the capture shows

1. **The three-way handshake.** Filter `tcp.flags.syn == 1`. SYN, SYN-ACK, ACK
   precede every session — the application protocols do nothing until TCP has
   established a connection.

2. **The server speaks first.** The `220` greeting arrives before the client
   has sent anything, which is unusual among client-server protocols.

3. **Application message versus TCP segment.** A message with a large
   attachment is visibly split across several TCP segments. The application
   sees one message; the network sees a byte stream cut into MSS-sized pieces
   and reassembled by TCP. This is exactly why `common/wire.py` must buffer
   partial lines: one `recv()` is not one protocol line.

4. **The password is readable.** Locate the `AUTH LOGIN` exchange and decode
   the base64 blob that follows a `334`:

   ```bash
   echo 'YXJ5YW4=' | base64 --decode      # -> aryan
   ```

   POP3's `PASS` command does not even use base64 — the password appears as
   plain text. **This is the central security finding of the project.** The
   remedy in real deployments is to run both protocols inside TLS: implicit
   TLS on ports 465 and 995, or `STARTTLS` upgrading the plaintext port in
   place.

## 8. Testing

The project ships an automated test suite of 47 tests, run with:

```bash
python3 -m unittest discover tests -v
```

| Group | What it verifies |
|---|---|
| Wire layer | Lines split across `recv()` calls reassemble correctly; a bare `\n` is *not* a line ending; dot-stuffing round-trips; over-long lines are rejected |
| MIME | Header folding and case-insensitivity; binary attachments survive a build/parse round trip; a malicious `../../` filename cannot escape the output directory |
| Accounts and store | Passwords are not stored in the clear; the maildrop lock is exclusive |
| End-to-end | Send → retrieve with body and attachment intact; a body containing a lone `.` line is not truncated; one message to two recipients is delivered twice |
| POP3 semantics | `DELE` without `QUIT` deletes nothing; `DELE` with `QUIT` commits; `RSET` unmarks; message numbers do not shift; `UIDL` is stable across sessions |
| Negative cases | `503` out of order, `500` unknown command, `501` bad syntax, `550` unknown mailbox, `535` bad credentials, `530` when auth is required |

In [18]:
import subprocess
result = subprocess.run(
    [sys.executable, "-m", "unittest", "discover", "tests"],
    cwd=PROJECT, capture_output=True, text=True)
print(result.stderr[-800:])

...............................................
----------------------------------------------------------------------
Ran 47 tests in 0.040s

OK



## 9. Results

| Requirement | Status |
|---|---|
| SMTP server implementing RFC 5321 on raw sockets | Implemented — 11 commands, full reply-code set |
| POP3 server implementing RFC 1939 on raw sockets | Implemented — 12 commands, all three states |
| Mail client speaking both protocols | Implemented — `SMTPClient` and `POP3Client` |
| Send and receive within localhost | Verified above and by `run_demo.py` |
| Multiple mailboxes with authentication | `AUTH LOGIN` / `PLAIN`, `USER`/`PASS`, `APOP` |
| Attachments | MIME `multipart/mixed`, built and parsed by hand |
| Protocol visibility | Every `C:`/`S:` line traced to console, file and GUI |
| Command-line interface | `client/cli.py` |
| Graphical interface | `client/gui.py` (Tkinter) |
| Automated tests | 47 tests, all passing |

Mail was sent from `aryan@localhost` to `prof@localhost` and to
`friend@localhost`, retrieved intact with headers, body and attachment
preserved, and deleted through the correct POP3 UPDATE-state semantics.

## 10. Limitations and future scope

| Limitation | What it would take to lift |
|---|---|
| **No TLS.** Passwords and message bodies cross the wire in the clear. | Wrap the sockets with Python's `ssl` module; add a `STARTTLS` capability to the `EHLO` list and a `STLS` command to POP3. This is the single most valuable addition. |
| **No relaying.** Every mailbox is local; there is no next hop. | Look the recipient's domain up in DNS for its **MX** record, then open an SMTP session to that host — this is what turns an isolated server into part of the global mail network. |
| **No queue or retry.** A delivery failure is permanent. | Spool accepted mail to a queue directory and retry on a backoff schedule, which is what a `4xx` temporary reply code is *for*. |
| **No spam or spoofing defence.** The envelope sender is unverified. | SPF (does DNS authorise this host to send for that domain?), DKIM (is the message cryptographically signed?), DMARC (what to do when those fail?). |
| **POP3 only.** Mail must be downloaded to be read. | Implement IMAP (RFC 3501) for server-side folders, partial fetch and multi-device access. |
| **Plaintext secret needed for APOP.** | Superseded by SASL mechanisms over TLS. |
| **Single-process maildrop lock.** The lock is held in memory, so it does not coordinate between separate server processes. | Use an on-disk lock file, or the Maildir naming convention, which is lock-free by design. |

## 11. Conclusion

A working email system was built from first principles and demonstrated
end-to-end on the loopback interface. Implementing both protocols directly on
sockets — rather than calling `smtplib` and `poplib` — made several things
concrete that are easy to read past in a textbook:

* **The reply code is a protocol, not decoration.** The difference between
  `500` (no such command) and `503` (right command, wrong moment) is the state
  machine expressing itself, and a client can act correctly on an unfamiliar
  code by reading only its first digit.

* **The envelope is not the headers.** SMTP routes on `MAIL FROM` and
  `RCPT TO` and never reads the `From:` line the recipient is shown. That gap
  is not a bug in any implementation — it is the design, and it is precisely
  where email spoofing lives.

* **`DELE` does not delete.** POP3's three-state model, with deletion deferred
  to UPDATE at `QUIT`, means an interrupted session loses nothing. This
  behaviour only becomes obvious once you have implemented it.

* **A TCP segment is not a protocol line.** The buffering in `common/wire.py`
  exists because one `recv()` may return half a line or three. Reading about
  "TCP is a byte stream, not a message stream" is not the same as having to
  write the code that copes with it.

* **Base64 is not encryption.** Watching a password appear in cleartext in a
  packet capture is a far more durable lesson than being told that SMTP was
  designed for a smaller and more trusting network than the one it now runs on.

The system satisfies CO5 in full: the SMTP and POP3 command sets, their reply
conventions, their state machines and their interaction through a shared mail
store were all implemented and demonstrated rather than merely described.

### References

1. RFC 5321 — *Simple Mail Transfer Protocol*
2. RFC 1939 — *Post Office Protocol version 3*
3. RFC 5322 — *Internet Message Format*
4. RFC 2045/2046 — *MIME Parts One and Two*
5. RFC 4954 — *SMTP Service Extension for Authentication*
6. A. S. Tanenbaum, *Computer Networks*, 5th ed., §7.2 — Electronic Mail
7. B. A. Forouzan, *Data Communications and Networking*, 5th ed., ch. 26

In [19]:
# Shut the servers down and remove the temporary mail store.
import shutil
smtp_server.stop()
pop_server.stop()
shutil.rmtree(workdir, ignore_errors=True)
print("Servers stopped; temporary mail store removed.")

Servers stopped; temporary mail store removed.
